# Оценка рейтингов фильмов


**Датасет:** MovieLens 20M — https://www.kaggle.com/datasets/grouplens/movielens-20m-dataset

**Постановка:** гибридное предсказание оценки `user → movie` (регрессия, `rating ∈ [0.5, 5.0]`).

**Метрики качества:** RMSE, MAE, R².

---

## 1. Введение и постановка задачи

### 1.1. Основные понятия

- **Рекомендательная система** — система, предсказывающая предпочтения пользователя
  (в нашем случае — оценку фильма).
- **Коллаборативная фильтрация (collaborative filtering)** — подход, использующий
  историю оценок множества пользователей: похожие пользователи похоже оценивают
  фильмы.
- **Контентная фильтрация (content-based)** — подход, опирающийся на признаки самих
  объектов (жанры, теги, год).
- **Гибридный подход** — объединяет коллаборативную и контентную информацию.
- **Эмбеддинги / латентные факторы (latent factors)** — обучаемые плотные векторы,
  кодирующие пользователей и фильмы в скрытом пространстве.
- **Матричная факторизация (Matrix Factorization)** — приближение матрицы оценок
  произведением матриц латентных факторов пользователей и фильмов.
- **Нейросетевая коллаборативная фильтрация (Neural CF / NeuMF)** — замена скалярного
  произведения эмбеддингов на нелинейную нейросеть (MLP).

### 1.2. Постановка задачи

Дана история оценок $(u, i, r)$, где $u$ — пользователь, $i$ — фильм,
$r \in \{0.5, 1.0, \dots, 5.0\}$ — оценка.

Требуется построить модель $\hat{r} = f(u, i, x_i)$, предсказывающую оценку, где
$x_i$ — контентные признаки фильма (жанры, genome-теги, год выпуска).

Это **задача регрессии**: целевая переменная — вещественная оценка в диапазоне
$[0.5, 5.0]$.

### 1.3. Подходы и методы решения

Рассматриваем последовательность моделей нарастающей сложности:

1. **Baseline** — глобальное среднее и модель смещений
   $\hat{r}_{ui} = \mu + b_u + b_i$ (точка отсчёта).
2. **Matrix Factorization (MF)** — эмбеддинги пользователя и фильма + смещения,
   скалярное произведение.
3. **NeuMF (Neural CF)** — конкатенация эмбеддингов user/movie, поданная в MLP.
4. **Hybrid** — эмбеддинги user/movie + отобранные контентные признаки фильма → MLP.

Сравнение этих методов между собой и с baseline — основная цель эксперимента.

### 1.4. Метрики оценки качества

- **RMSE** (Root Mean Squared Error) — основная метрика, сильнее штрафует крупные
  ошибки:
  $$\mathrm{RMSE} = \sqrt{\frac{1}{N}\sum_{(u,i)} (r_{ui} - \hat{r}_{ui})^2}.$$
- **MAE** (Mean Absolute Error) — средняя абсолютная ошибка, устойчивее к выбросам:
  $$\mathrm{MAE} = \frac{1}{N}\sum_{(u,i)} |r_{ui} - \hat{r}_{ui}|.$$
- **R²** (коэффициент детерминации) — доля объяснённой дисперсии целевой переменной.

Качество измеряем на отложенной (test) выборке, не участвующей в обучении и подборе
гиперпараметров.

## 2. Импорт библиотек и конфигурация

In [ ]:
# Импорт библиотек: numpy, pandas, torch (nn, DataLoader),
# sklearn (метрики, отбор признаков), matplotlib.


In [ ]:
# Конфигурация эксперимента:
# - фиксация random seed (numpy, torch) для воспроизводимости;
# - выбор device (cuda, если доступен, иначе cpu);
# - пути к каталогу data/ и файлам датасета;
# - параметры подвыборки и порогов активности (min оценок на пользователя/фильм);
# - гиперпараметры по умолчанию (размер эмбеддинга, batch_size, lr, epochs и т.п.).


## 3. Загрузка данных

Состав датасета MovieLens 20M:

| Файл | Содержание | Используем |
|------|-----------|-----------|
| `rating.csv` | userId, movieId, rating, timestamp (≈20 млн строк) | да — целевые оценки |
| `movie.csv` | movieId, title, genres | да — жанры, год из названия |
| `genome_scores.csv` | movieId, tagId, relevance (≈1128 тегов) | да — контентные признаки |
| `genome_tags.csv` | tagId, tag | да — расшифровка тегов |
| `tag.csv` | пользовательские текстовые теги | нет (опционально) |
| `link.csv` | связи с IMDb/TMDb | нет |

Из-за объёма `rating.csv` (~690 МБ) работаем с **подвыборкой активных
пользователей** (~1–2 млн оценок).

In [ ]:
# Загрузка справочников: movie.csv, genome_scores.csv, genome_tags.csv.


In [ ]:
# Загрузка rating.csv с подвыборкой активных пользователей:
# - chunked-чтение или фильтр по числу оценок на пользователя;
# - контроль итогового размера подвыборки (~1-2 млн строк);
# - приведение timestamp к datetime.


## 4. Разведочный (описательный) анализ данных — EDA

Цель — понять структуру и качество данных: распределение оценок, активность
пользователей и фильмов, разреженность матрицы, временные и жанровые эффекты,
характер genome-признаков. Выводы EDA определяют решения по предобработке и
выбору моделей.

In [ ]:
# Базовые статистики: размеры выборки, число уникальных пользователей и фильмов,
# разреженность матрицы оценок (доля заполненных ячеек), describe() по рейтингам.


In [ ]:
# Распределение оценок: гистограмма rating, доли по каждому значению (0.5...5.0).


In [ ]:
# Активность: распределение числа оценок на пользователя и на фильм
# (длинный хвост, лог-шкала).


In [ ]:
# Временной разрез: динамика числа оценок и средней оценки по времени (timestamp).


In [ ]:
# Анализ жанров: частоты жанров, средняя оценка и число оценок по жанрам.


In [ ]:
# Обзор genome-признаков: распределение релевантности тегов,
# самые информативные/вариативные теги, при желании — корреляции.


### Выводы по EDA

_(Заполняется по результатам: дисбаланс оценок, длинный хвост активности,
разреженность, влияние жанров/года — и как это учтём в предобработке и моделях.)_

## 5. Предобработка данных

План и обоснование шагов предобработки перед обучением.

In [ ]:
# Фильтрация неактивных пользователей и редко оцениваемых фильмов
# по порогам из конфигурации (борьба с разреженностью и шумом).


In [ ]:
# Переиндексация идентификаторов: userId/movieId -> непрерывные индексы [0..N)
# для слоёв эмбеддингов; сохранение словарей соответствия.


In [ ]:
# Формирование контентных признаков фильма:
# - one-hot кодирование жанров;
# - извлечение года из названия (regex) и нормализация;
# - матрица genome-релевантности (movie x tag).


In [ ]:
# Разбиение на train / val / test (по времени либо стратифицированно по пользователю).
# Контроль утечек; фиксация cold-start случаев (новые user/movie в test).


## 6. Отбор значимых признаков

Выполнение требования задания «отбор значимых признаков». Применяется к контентным
признакам фильма (жанры, ~1128 genome-тегов, год). Цель — оставить наиболее
информативные признаки, снизить размерность и шум перед подачей в гибридную модель.

In [ ]:
# Удаление низковариативных признаков (VarianceThreshold).


In [ ]:
# Оценка значимости признаков относительно целевой переменной:
# корреляция / mutual information; отбор top-K genome-тегов.


In [ ]:
# Визуализация важности признаков (топ по значимости) и фиксация
# финального набора контентных признаков для гибридной модели.


### Итоговый набор признаков

_(Список отобранных признаков и краткое обоснование выбора.)_

## 7. Проектирование и реализация исследовательского стенда

Единый стенд обеспечивает честное сравнение моделей: общий формат данных
(`Dataset`/`DataLoader`), единый цикл обучения с ранней остановкой, единые функции
метрик и общий интерфейс оценки. Все модели обучаются и оцениваются одинаково.

In [ ]:
# Класс Dataset и DataLoader: батчи (user_idx, movie_idx, content_features) -> rating.


In [ ]:
# Функции метрик: RMSE, MAE, R² (единые для всех моделей).


In [ ]:
# Универсальные процедуры train_model() и evaluate_model():
# цикл по эпохам, оптимизатор, логирование loss/RMSE, early stopping,
# сохранение кривых обучения.


In [ ]:
# Baseline-предсказатели как точка отсчёта:
# - глобальное среднее (mu);
# - модель смещений (mu + b_user + b_movie).


## 8. Нейросетевые модели

Три архитектуры нарастающей сложности. Различие — в том, как используется информация:
от чистой коллаборативной (MF, NeuMF) к гибридной (контент + взаимодействия).

In [ ]:
# Модель 1 — Matrix Factorization:
# эмбеддинги user и movie + смещения b_u, b_i; предсказание = dot(p_u, q_i) + b_u + b_i + mu.


In [ ]:
# Модель 2 — NeuMF (Neural Collaborative Filtering):
# конкатенация эмбеддингов user и movie -> MLP (несколько слоёв с нелинейностью) -> оценка.


In [ ]:
# Модель 3 — Hybrid:
# эмбеддинги user/movie + отобранные контентные признаки фильма (раздел 6) -> MLP -> оценка.


## 9. Подбор гиперпараметров

Небольшой поиск (grid/random search) по валидационной выборке. Пространство поиска:
размер эмбеддинга, learning rate, число и ширина скрытых слоёв, dropout, weight decay.
Для каждой модели выбираем лучшую конфигурацию по RMSE на валидации.

In [ ]:
# Цикл подбора гиперпараметров для каждой модели:
# перебор конфигураций, обучение на train, оценка на val,
# сводная таблица результатов и выбор лучших гиперпараметров.


## 10. Обучение финальных моделей

In [ ]:
# Обучение baseline и трёх моделей с лучшими гиперпараметрами на train;
# сохранение истории обучения (loss/RMSE по эпохам).


In [ ]:
# Графики сходимости (train/val) для каждой модели.


## 11. Оценка качества и сравнительный анализ

Финальное сравнение моделей проводится на отложенной test-выборке, не участвовавшей
ни в обучении, ни в подборе гиперпараметров.

In [ ]:
# Сводная таблица метрик (RMSE / MAE / R²) по всем моделям и baseline.


In [ ]:
# Столбчатые диаграммы сравнения метрик между моделями.


In [ ]:
# Анализ остатков лучшей модели: распределение ошибок,
# ошибки по сегментам (активные/неактивные пользователи, популярные/редкие фильмы).


## 12. Выбор лучшей модели и анализ результатов

_(Обоснование выбора лучшей модели по метрикам; интерпретация: что дали контентные
признаки и их отбор, на каких объектах модели ошибаются, проблема cold-start.)_

## 13. Выводы

_(Итоги по решению задачи; сравнение методов; ограничения подхода и направления
дальнейшего развития.)_